<a href="https://colab.research.google.com/github/wlgns222/ROKA/blob/main/ai-study/deep-learning-from-scratch-vol1/Ch6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ch6. Training Techniques

## 6.1 Optimization

### 6.1.2 확률적 경사 하강법 (SGD)

**SGD 란**

기울어진 방향으로 일정 거리만큼을 가겠다는 방법

In [ ]:
class SGD :
  def __init__(self, lr = 0.01) :
    self.lr = lr
  def update(self, params, grads):
    for key in params.keys():
      params[key] -= self.lr * grads[key]

- 인수 lr : learning rate 학습률
- 메서드 update(params, grads)
  - 인수 params, grads : params['W1'], grads['W1'] 등과 같이 매개변수와 기울기를 저장

SGD 클래스를 사용하면 신경망 매개변수 진행을 다음과 같이 수행 가능하다.

In [ ]:
network = TwoLayerNet(...)
optimizer = SGD()

for i in range(10000) :
  ...
  x_batch, t_batch = get_mini_batch(...)
  grads = network.gradient(x_batch, t_batch)
  params = network.params

  optimizer.update(params, grads)
  ...

**optimizer : 최적화를 행하는 자**

최적화를 담당하는 클래스를 분리해 구현하면 기능을 모듈화 하기 좋다.

Momentum 이라는 최적화 기법 역시 update(params, grads) 라는 메서드를 갖도록 구현하고, optimizer = SGD() 문장을 optimizer = Momentum()으로 변경하면 간단하게 바꿀 수 있다.

### 6.1.3 SGD의 단점

SGD는 단순하고 구현도 쉬우나, 문제에 따라서 비효율적일 때가 있다.

<br>


**SGD는 비등방성(anisotropy) 함수에서 비효율적이다.**

비등방성이란 방향에 따라 물리적 성질이 바뀌는 것이다. 즉, 기울기가 가르키는 지점이 하나(최솟값)가 아니라 여러가지일 때 기울기가 가르키는 방향으로 이동하는 SGD는 비효율적인 움직임을 보인다.


SGD의 이런 단점을 개선해주는 optimizer
- Momentum
- AdaGrad
- Adam

### 6.1.4 모멘텀 (Momentum)

**모멘텀 (Momentum) 이란**

'운동량'을 뜻하는 단어로 물리와 관계가 있다. 모멘텀 기법은 수식으로 다음과 같이 쓸 수 있다.

1. $v \leftarrow \alpha v - \eta \frac{\partial L}{\partial W}$

2. $W \leftarrow W + v$

SGD 에서와 마찬가지로 $W$ 는 갱신할 매개변수, $\frac{\partial L}{\partial W}$ 은 $W$ 에대한 손실함수의 기울기, $\eta$ 는 학습률이다.

$v$ 라는 변수가 새로 나오는데, 이는 물리에서의 속도에 해당한다.

$\alpha$ 는 마찰력같은 존재로 보통 0.9로 설정하여 과거의 속도를 어느정도 유지하며 가속도를 붙게한다.

**모멘텀은 공이 기울기를 따라 구르듯 움직인다**



In [ ]:
import numpy as np

class Momentum :
  def __init__ (self, lr = 0.01, momentum = 0.9):
    self.lr = lr
    self.momentum = momentum
    self.v = None
  def update(self, params, grads) :
    #Initialize v buffer on the first run
    if self.v is None :
      self.v = {}
      for key, val in params.items():
        self.v[key] = np.zero_like(val)
    for key in params.keys():
      # v = alpha * v - lr * grads
      self.v[key] = self.momentum * self.v[key] - self.lr * grads[key]
      #W = W + v
      params[key] += self.v[key]

**모멘텀의 장점**
1. 지그재그로 요동치는 구간에서 좌우 움직임은 서로 상쇄한다.
2. 목표를 향한 전진방향에 가속도를 붙임으로서 더 빠르게 손실함수의 바닥에 도달할 수 있다.

### 6.1.5 아다그라드 (AdaGrad)

신경망 학습에서 학습률 값이 매우 중요하다. 값이 너무 작으면 학습 시간이 길어지며, 너무 크면 발산하여 학습이 제대로 이뤄지지 않는다.

<br>

**아다그라드 (AdaGrad) 의 핵심 : '학습률 감소'**

아다그라드는 '학습률 감소'를 매개변수마다 다르게 적용하여, 각각의 매개변수에 맞춤형 값을 제공한다. 많이 움직인 매개변수는 학습률을 대폭 줄여서 세밀하게 조정하고, 적게 움직인 매개변수는 학습률을 유지하여 더 공부할 기회를 준다. 이를 수식으로 나타내면 다음과 같다.

1. $h \leftarrow h + (\frac{\partial L}{\partial W})^2$

2. $W \leftarrow W - \eta \frac{1}{\sqrt{h}} \frac{\partial L}{\partial W}$

마찬가지로 $W$ 는 갱신할 매개변수, $\frac{\partial L}{\partial W}$ 은 $W$ 에대한 손실함수의 기울기, $\eta$ 는 학습률이다.

$h$ 는 기존 기울기 값을 제곱하여 계속 더해준다. 그리고 매개변수를 갱신할 때 $\frac{1}{\sqrt{h}}$ 를 곱하여 학습률을 조정한다.


In [ ]:
class AdaGrad :
  def __init__ (self, lr = 0.01) :
    self.lr = lr
    self.h = None
  def update (self, params, grads) :
    if self.h is None :
      self.h = {}
      for key, val in params.items() :
        self.h[key] = np.zeros_like(val)
    for key in params.keys() :
      #Square of current gradient
      self.h[key] += grads[key] * grads[key]
      #Update params : divide by sqrt(h)
      #Add 1e-7 to avoid division by zero
      params[key] -= self.lr * grads[key] / (np.sqrt(self.h[key]) + 1e-7)

**Note**

AdaGrad 는 과거의 기울기를 제곱하여 계속 더해감으로 $h$ 가 무한히 커지고 학습률이 0이 되어 '학습정지' 상태에 빠질 수 있다는 단점이 있다.

이를 보완하기 위해 **RMSProps** 라는 방법이 있다. 이는 먼 과거의 기울기는 잊고 새로운 기울기 정보를 크게 반영하는 기법이다.

### 6.1.6 아담 (Adam)

**아담 (Adam) 이란**

*Momentum* 의 장점과 *AdaGrad* 의 장점을 결합한 기법이다.

2015년에 제안된 방법이며, 현대 딥러닝 프로젝트에서 가장 많이 쓰이는 범용성이 뛰어난 엔진이다.

아담의 특징으로 **편향 보정** (Bias Correction) 이 있다.

학습 초기에는 누적된 데이터가 없어 속도나 학습률 조절이 부정확할 수 있는데, 이를 수학적으로 보정하여 첫 번째 스텝부터 엔진이 최고 출력을 낼 수 있도록 돕는 방법이다.

**Note**

Adam 은 3개의 하이퍼파라미터를 사용한다.
1. 학습률 - $α$
2. 일차 모멘텀용 계수 - $\beta_1=0.9$
3. 이차 모멘텀용 계수 - $\beta_2=0.999$